# Seed-VC real-time tiny の fine-tune

Google Colab と Kaggle の GPU ノートブックで、Seed-VC の real-time tiny モデルを話者別に fine-tune するためのノートブックです。

対象話者の明示的な同意を得た音声だけを使用してください。
このノートブックは V2 や offline 用モデルを学習しません。
生成する `ft_model.pth` は、このリポジトリの `Seed-VC Fine-tuned (realtime / higher similarity)` プロファイル用です。

実行前に、Colab では「ランタイム」から GPU を有効にします。
Kaggle では「Notebook options」から Accelerator を GPU にし、インターネット接続も有効にします。

In [ ]:
import platform
import subprocess

print(platform.platform())
subprocess.run(["nvidia-smi"], check=True)

## 学習データ

各音声ファイルは 1〜30 秒、単一話者、BGM なし、クリッピングなしにしてください。
公式上は話者あたり 1 発話から学習できますが、品質確認の開始値としては 5〜15 分を 20〜60 ファイルに分割する構成を推奨します。

Colab では Drive をマウントして `DATASET_DIR` を変更します。
Kaggle では音声 Dataset を notebook に添付し、`/kaggle/input/...` の実際のパスに変更します。

In [ ]:
from pathlib import Path

IS_KAGGLE = Path("/kaggle").exists()
WORK_DIR = Path("/kaggle/working" if IS_KAGGLE else "/content")

if IS_KAGGLE:
    # 例: Path("/kaggle/input/my-voice-dataset/target-voice")
    DATASET_DIR = Path("/kaggle/input/CHANGE_ME")
else:
    from google.colab import drive
    drive.mount("/content/drive")
    # 例: Path("/content/drive/MyDrive/seedvc/target-voice")
    DATASET_DIR = Path("/content/drive/MyDrive/CHANGE_ME")

if "CHANGE_ME" in str(DATASET_DIR) or not DATASET_DIR.is_dir():
    raise ValueError(f"DATASET_DIR を実在する音声ディレクトリに変更してください: {DATASET_DIR}")

print(f"platform: {'Kaggle' if IS_KAGGLE else 'Colab'}")
print(f"dataset: {DATASET_DIR}")

In [ ]:
import sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "soundfile"], check=True)
import soundfile as sf

SUPPORTED_EXTENSIONS = {".wav", ".flac", ".mp3", ".m4a", ".opus", ".ogg"}
audio_files = sorted(path for path in DATASET_DIR.rglob("*") if path.suffix.lower() in SUPPORTED_EXTENSIONS)
if not audio_files:
    raise ValueError(f"対応する音声ファイルがありません: {DATASET_DIR}")

invalid_files = []
durations = []
for path in audio_files:
    try:
        info = sf.info(path)
        duration = info.frames / info.samplerate
    except RuntimeError as exc:
        invalid_files.append((path, f"読めません: {exc}"))
        continue
    durations.append(duration)
    if not 1.0 <= duration <= 30.0:
        invalid_files.append((path, f"{duration:.2f} 秒"))

if invalid_files:
    details = '\n'.join(f"{path}: {reason}" for path, reason in invalid_files[:20])
    raise ValueError("1〜30 秒ではない、または読めない音声があります。修正してから続けてください。\n" + details)

print(f"files: {len(audio_files)}")
print(f"total duration: {sum(durations) / 60:.2f} minutes")
print(f"duration range: {min(durations):.2f}–{max(durations):.2f} seconds")

## Seed-VC のインストール

このリポジトリで検証したコミットを取得し、Python 3.10 の仮想環境へ依存関係を入れます。
Seed-VC 公式 requirements の CUDA nightly 指定は安定版 Torch 指定と競合するため除外します。

In [ ]:
SEEDVC_REVISION = "51383efd921027683c89e5348211d93ff12ac2a8"
SEEDVC_DIR = WORK_DIR / "seed-vc"

if not SEEDVC_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/Plachtaa/seed-vc.git", str(SEEDVC_DIR)], check=True)
subprocess.run(["git", "checkout", SEEDVC_REVISION], cwd=SEEDVC_DIR, check=True)
subprocess.run(["python", "-m", "pip", "install", "-q", "uv"], check=True)
subprocess.run(["uv", "venv", "--python", "3.10", ".venv"], cwd=SEEDVC_DIR, check=True)

requirements = (SEEDVC_DIR / "requirements.txt").read_text().splitlines()
filtered_requirements = [line for line in requirements if "nightly/cu126" not in line]
requirements_path = WORK_DIR / "seedvc-requirements.txt"
requirements_path.write_text("\n".join(filtered_requirements) + "\n")
python_path = SEEDVC_DIR / ".venv/bin/python"
subprocess.run(["uv", "pip", "install", "-p", str(python_path), "-r", str(requirements_path)], check=True)
subprocess.run(["uv", "pip", "install", "-p", str(python_path), "-U", "FreeSimpleGUI>=5.2"], check=True)
subprocess.run([str(python_path), "-c", "import torch; assert torch.cuda.is_available(); print(torch.__version__, torch.cuda.get_device_name(0))"], check=True)

## 学習設定

6GB では `BATCH_SIZE = 1` から始めます。
16GB 以上で余裕を確認できた場合だけ 2 に増やしてください。
`MAX_STEPS = 100` は動作確認用です。
品質を比較する本実行には 1000 を使います。

In [ ]:
RUN_NAME = "target-voice-realtime"
BATCH_SIZE = 1
MAX_STEPS = 1000
SAVE_EVERY = 250

if BATCH_SIZE < 1:
    raise ValueError("BATCH_SIZE は 1 以上にしてください。")
if MAX_STEPS < 1:
    raise ValueError("MAX_STEPS は 1 以上にしてください。")
print({"run_name": RUN_NAME, "batch_size": BATCH_SIZE, "max_steps": MAX_STEPS})

In [ ]:
config_path = SEEDVC_DIR / "configs/presets/config_dit_mel_seed_uvit_xlsr_tiny.yml"
train_command = [
    str(python_path), "train.py",
    "--config", str(config_path),
    "--dataset-dir", str(DATASET_DIR),
    "--run-name", RUN_NAME,
    "--batch-size", str(BATCH_SIZE),
    "--max-steps", str(MAX_STEPS),
    "--max-epochs", "1000",
    "--save-every", str(SAVE_EVERY),
    "--num-workers", "0",
]
subprocess.run(train_command, cwd=SEEDVC_DIR, check=True)

## モデルの書き出し

学習終了後、checkpoint と学習に用いた YAML を同梱した zip を作ります。
zip をダウンロードし、ローカルの Voice_Changer リポジトリで `models/seedvc-finetuned/` に展開してください。

In [ ]:
import shutil

run_dir = SEEDVC_DIR / "runs" / RUN_NAME
checkpoint_path = run_dir / "ft_model.pth"
trained_config_path = run_dir / config_path.name
if not checkpoint_path.is_file() or not trained_config_path.is_file():
    raise FileNotFoundError(f"学習成果物が不足しています: {run_dir}")

export_dir = WORK_DIR / "seedvc-finetuned"
export_dir.mkdir(exist_ok=True)
shutil.copy2(checkpoint_path, export_dir / "ft_model.pth")
shutil.copy2(trained_config_path, export_dir / "config_dit_mel_seed_uvit_xlsr_tiny.yml")
archive_path = shutil.make_archive(str(WORK_DIR / "seedvc-finetuned"), "zip", root_dir=export_dir)
print(f"export: {archive_path}")
print("ローカルでは zip 内の 2 ファイルを models/seedvc-finetuned/ に配置してください。")

In [ ]:
if not IS_KAGGLE:
    from google.colab import files
    files.download(archive_path)
else:
    print(f"Kaggle の Output からダウンロードしてください: {archive_path}")